In [5]:
import pandas as pd
df = pd.read_csv("ASWSINDEXEOD_202605210941.csv")
df['TRADE_DT'] = pd.to_datetime(df['TRADE_DT'].astype(str))

In [6]:
level1_dict = {
    '801010.SI': '农林牧渔',
    '801020.SI': '采掘',
    '801030.SI': '化工',
    '801040.SI': '钢铁',
    '801050.SI': '有色金属',
    '801080.SI': '电子',
    '801110.SI': '家用电器',
    '801120.SI': '食品饮料',
    '801130.SI': '纺织服饰',
    '801140.SI': '轻工制造',
    '801150.SI': '医药生物',
    '801160.SI': '公用事业',
    '801170.SI': '交通运输',
    '801180.SI': '房地产',
    '801200.SI': '商贸零售',
    '801210.SI': '社会服务',
    '801230.SI': '综合',
    '801710.SI': '建筑材料',
    '801720.SI': '建筑装饰',
    '801730.SI': '电力设备',
    '801740.SI': '国防军工',
    '801750.SI': '计算机',
    '801760.SI': '传媒',
    '801770.SI': '通信',
    '801780.SI': '银行',
    '801790.SI': '非银金融',
    '801880.SI': '汽车',
    '801890.SI': '机械设备',
    '801950.SI': '煤炭',
    '801960.SI': '石油石化',
    '801970.SI': '环保',
    '801980.SI': '美容护理'
}
# 筛选一级行业
df_level1 = df[
    df['S_INFO_WINDCODE'].isin(level1_dict.keys())
].copy()
df_level1['industry_name'] = (
    df_level1['S_INFO_WINDCODE']
    .map(level1_dict)
)
# 排序
df_level1 = df_level1.sort_values(
    ['S_INFO_WINDCODE', 'TRADE_DT']
)
#计算每日收益率
df_level1['return'] = (
    df_level1
    .groupby('S_INFO_WINDCODE')['S_DQ_CLOSE']
    .pct_change()
)

In [7]:
# 保存为宽表：保留日期列，每个行业一列，按相同交易日期对齐
return_wide = df_level1.pivot(
    index='TRADE_DT',
    columns='industry_name',
    values='return'
).sort_index()

# reset_index 后 TRADE_DT 会作为第一列保留下来
return_wide = return_wide.reset_index()
return_wide.columns.name = None

return_wide.to_feather("申万一级行业_with_dailyreturn.feather")